# 19.8 回归断点设计 / Regression Discontinuity Design (RDD)

**中文**：很多处理是**按一个阈值分配**的:考试分 ≥60 就发奖学金、收入 <某线就享补贴、得票率 >50% 就当选、AUM ≥某额度就升 VIP。**回归断点(RDD)** 抓住这个规则的一个绝妙特性:**卡在阈值两侧的人几乎一模一样**——59 分和 61 分的学生,能力、家境、努力都差不多,唯一的系统差异就是"一个拿到了奖学金,一个没拿到"。于是**阈值附近就是一个天然的随机实验**!这是**准确度最高、最可信**的观测因果方法(接近 RCT)。
**English**: Many treatments are **assigned by a threshold**: score ≥60 → scholarship, income < a line → subsidy, vote share >50% → win, AUM ≥ a level → VIP. **Regression Discontinuity (RDD)** exploits a beautiful feature of this rule: **people just on either side of the threshold are nearly identical** — students scoring 59 vs 61 have similar ability, background, effort; the only systematic difference is "one got the scholarship, one didn't." So **near the threshold is a natural randomized experiment**! It is the **most accurate and credible** observational causal method (closest to an RCT).

---

**中文**：设定:一个**驱动变量(running variable)$X$**(如考试分),一个**阈值 $c$**;当 $X\ge c$ 时接受处理。有两种:
**English**: Setup: a **running variable $X$** (e.g. test score), a **cutoff $c$**; treatment when $X\ge c$. Two kinds:
- **精确断点(Sharp RDD)**:处理是 $X$ 的**确定性阶跃**——$X\ge c$ 一定处理,否则一定不处理。因果效应 = 结果在阈值处的**跳变(jump)**。
  **Sharp RDD**: treatment is a **deterministic step** in $X$ — $X\ge c$ always treated, else never. The causal effect = the **jump** in the outcome at the cutoff.
- **模糊断点(Fuzzy RDD)**:跨过阈值只是**提高了接受处理的概率**(不是 100%,如"够分但没去领奖学金")。此时把"是否跨阈值"当**工具变量**(19.7),用跳变比来估。
  **Fuzzy RDD**: crossing the cutoff only **raises the probability** of treatment (not 100%, e.g. "eligible but didn't claim"). Then treat "crossing the cutoff" as an **instrument** (19.7), estimating via the ratio of jumps.

**中文**：估计方法:在阈值**两侧各拟合一条局部回归线**(通常用阈值附近一个**带宽(bandwidth)** 内的数据做局部线性回归),量出两条线在 $c$ 处的**垂直落差**——那就是因果效应。**关键假设=连续性/无操纵**:人们**不能精确控制**自己的 $X$ 恰好卡在阈值(如学生不能把分数精确刷到 60),否则阈值两侧就不再可比了。
**English**: Estimation: **fit a local regression on each side** of the cutoff (usually a local linear regression within a **bandwidth** around it) and measure the **vertical gap** between the two lines at $c$ — that is the causal effect. **Key assumption = continuity / no manipulation**: people **cannot precisely control** their $X$ to land just at the cutoff (a student can't dial their score to exactly 60), else the two sides are no longer comparable.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 准实验必考）**
> **中文**：**RDD=按阈值分配处理→阈值两侧as-good-as-random**(59分vs61分几乎一样), 效应=结果在阈值处的**跳变**。**Sharp**(确定性阶跃)vs **Fuzzy**(跨阈值改变处理概率→当IV)。估计=阈值两侧**局部线性回归**在 c 处的落差(选**带宽**:小→偏差小方差大, 大→反之)。**核心假设=连续性/无操纵**:不能精确控制驱动变量卡阈值——用 **McCrary 密度检验**看阈值处样本密度有无跳变(有跳变=有操纵=RDD失效)。**内部效度极高(近RCT), 但只估阈值处的局部效应(LATE at cutoff), 外推性差**。协变量在阈值处应连续(安慰剂检验)。经典:近距离选举(现任优势)、班级规模(Maimonides规则)、奖学金分数线。
> **English**: **RDD = threshold-assigned treatment → both sides of the cutoff are as-good-as-random** (score 59 vs 61 are nearly identical); the effect = the **jump** in the outcome at the cutoff. **Sharp** (deterministic step) vs **Fuzzy** (crossing changes treatment probability → use as an IV). Estimate = the gap at c from **local linear regressions** on each side (choose a **bandwidth**: small → low bias, high variance; large → the reverse). **Core assumption = continuity / no manipulation**: units can't precisely control the running variable to land at the cutoff — check with the **McCrary density test** (a jump in sample density at the cutoff = manipulation = RDD fails). **Very high internal validity (near-RCT) but only the local effect at the cutoff (LATE at cutoff), poor external validity**. Covariates should be continuous at the cutoff (placebo test). Classics: close elections (incumbency advantage), class size (Maimonides' rule), scholarship score lines.


In [ ]:

# ============================================================
# 模拟:分数≥60 发奖学金, 看奖学金对后续GPA的因果效应 / scholarship if score≥cutoff → effect on later GPA
# 中文:驱动变量=入学考试分 X, 阈值 60。结果 Y(如毕业GPA)本身随分数平滑上升(高分学生本来就更强),
#      奖学金带来真实跳变 +8。朴素比较会被"分数本身的作用"污染, RDD 只看阈值处的跳变。
# English: running variable = entry score X, cutoff 60. Outcome Y (e.g. final GPA) rises smoothly with score
#      (high scorers are stronger anyway); the scholarship causes a true jump of +8. Naive comparison is
#      confounded by "score's own effect"; RDD isolates the jump at the cutoff.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
N=3000; cutoff=60; true_jump=8.0
X=rng.uniform(20,100,N)                                      # 入学考试分 / entry score
treated=(X>=cutoff).astype(int)                             # 精确断点:≥60 发奖学金 / sharp assignment
Y=30 + 0.5*X + 0.004*(X-60)**2 + true_jump*treated + rng.normal(0,5,N)   # 结果 / outcome

naive = Y[treated==1].mean() - Y[treated==0].mean()        # 朴素:全体处理-对照 / naive comparison
print(f"真实奖学金效应(跳变)/ true scholarship effect: +{true_jump}")
print(f"朴素比较(全体拿奖-没拿)/ naive: +{naive:.1f}  (被'分数本身的作用'严重污染 → 高估)")


**中文**：朴素比较给出 **+30 多**——因为拿奖学金的都是高分学生,他们的 GPA 本来就高,这份功劳被错算给了奖学金。**RDD** 的做法:只看**阈值 60 附近**的学生(他们分数几乎一样、可比),在两侧各拟合一条局部线性回归,量出在 60 处的垂直落差。
**English**: The naive comparison gives **+30+** — because scholarship recipients are all high scorers whose GPA is already high, and that credit is misattributed to the scholarship. **RDD's** approach: look only at students **near the cutoff 60** (nearly identical scores, comparable), fit a local linear regression on each side, and measure the vertical gap at 60.


In [ ]:

# ============================================================
# 从零实现 Sharp RDD 局部线性回归 / sharp RDD via local linear regression
# ============================================================
def rdd_estimate(X, Y, cutoff, bandwidth):
    m = np.abs(X-cutoff) <= bandwidth                       # 带宽内的样本 / within bandwidth
    def local_intercept_at_cutoff(xx, yy):                  # 局部线性回归在阈值处的截距 / fit, value at c
        A = np.c_[np.ones(len(xx)), xx-cutoff]              # 中心化到阈值 / center at cutoff
        beta = np.linalg.lstsq(A, yy, rcond=None)[0]
        return beta[0]                                      # 截距=在阈值处的预测值 / value at cutoff
    left  = local_intercept_at_cutoff(X[m&(X<cutoff)],  Y[m&(X<cutoff)])   # 左侧线在 c 的值 / left limit
    right = local_intercept_at_cutoff(X[m&(X>=cutoff)], Y[m&(X>=cutoff)])  # 右侧线在 c 的值 / right limit
    return right-left, left, right

est, left_val, right_val = rdd_estimate(X, Y, cutoff, bandwidth=10)
print(f"RDD 估计(带宽=10)/ RDD estimate: +{est:.2f}  ← 还原真值 {true_jump}!")
print(f"阈值处:左侧线 GPA={left_val:.1f}, 右侧线 GPA={right_val:.1f}, 落差={est:.1f}")
# 带宽敏感性 / bandwidth sensitivity
print("\n带宽敏感性 / bandwidth sensitivity:")
for h in [5,10,15,25,40]:
    e,_,_=rdd_estimate(X,Y,cutoff,h); print(f"  带宽 {h:2d}: 估计 +{e:.2f}")


**中文**：RDD 精确还原了真实效应 +8(朴素比较错到 +30)。带宽的选择是个权衡:**太小**→只用阈值附近极少数据,偏差小但方差大(不稳);**太大**→用了远离阈值的数据,那里两侧不再可比,引入偏差。下面可视化经典的 RDD 断点图。
**English**: RDD exactly recovers the true effect +8 (naive is off at +30). Bandwidth choice is a trade-off: **too small** → uses very few points near the cutoff, low bias but high variance (unstable); **too large** → uses points far from the cutoff where the sides are no longer comparable, adding bias. Below, the classic RDD discontinuity plot.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,4.7))
# ① 经典 RDD 断点图 / the classic RDD plot
sub=rng.choice(N,800,replace=False)
ax[0].scatter(X[sub],Y[sub],c=np.where(treated[sub]==1,"#C44E52","#4C72B0"),s=8,alpha=0.4)
# 两侧局部拟合线 / fitted lines on each side (bandwidth region)
h=10
for side,cond,color in [("左",X<cutoff,"#4C72B0"),("右",X>=cutoff,"#C44E52")]:
    mm=(np.abs(X-cutoff)<=h)&cond; xs=X[mm]; A=np.c_[np.ones(mm.sum()),xs-cutoff]; b=np.linalg.lstsq(A,Y[mm],rcond=None)[0]
    xr=np.linspace(xs.min(),xs.max(),50); ax[0].plot(xr,b[0]+b[1]*(xr-cutoff),color=color,lw=3)
ax[0].axvline(cutoff,ls="--",color="k",label="阈值 cutoff=60")
ax[0].annotate("",xy=(cutoff,right_val),xytext=(cutoff,left_val),arrowprops=dict(arrowstyle="<->",color="green",lw=2))
ax[0].text(cutoff+2,(left_val+right_val)/2,f"跳变\n+{est:.1f}",color="green",fontsize=10)
ax[0].set_title("RDD 断点图:阈值处的跳变=因果效应 / the jump at cutoff"); ax[0].set_xlabel("入学分数 X"); ax[0].set_ylabel("毕业 GPA"); ax[0].legend(fontsize=8)
# ② 带宽敏感性 / bandwidth sensitivity
hs=np.arange(4,45,2); ests=[rdd_estimate(X,Y,cutoff,h)[0] for h in hs]
ax[1].plot(hs,ests,"o-",color="#55A868"); ax[1].axhline(true_jump,ls="--",color="k",label="真值 +8")
ax[1].set_title("带宽敏感性:太大→有偏 / bandwidth sensitivity"); ax[1].set_xlabel("带宽 bandwidth"); ax[1].set_ylabel("RDD 估计"); ax[1].legend(fontsize=8)
# ③ 操纵检验:驱动变量的密度在阈值处应连续 / manipulation check: density continuous at cutoff
ax[2].hist(X,bins=40,color="#4C72B0",alpha=0.7); ax[2].axvline(cutoff,ls="--",color="k")
ax[2].set_title("McCrary 密度检验:阈值处密度无跳变=无操纵 / no manipulation"); ax[2].set_xlabel("驱动变量 X"); ax[2].set_ylabel("样本数")
plt.tight_layout(); plt.savefig("/tmp/ci08_viz.png",dpi=80); plt.show()
print(f"RDD +{est:.1f} vs 真值 +{true_jump} vs 朴素 +{naive:.1f} —— 只有 RDD 抓对了阈值处的因果跳变")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **RDD 是最接近随机实验的观测方法**:朴素比较错到 +30(被分数本身的作用污染),而 RDD 精确抓住阈值处的 +8 跳变。它的可信度来自一个无可辩驳的逻辑——**59 分和 61 分的人几乎没有系统差异**,唯一的区别就是处理。所以在阈值的**局部**,处理近乎随机分配。这让 RDD 的内部效度极高,在政策评估、教育、金融监管里被高度信赖。
2. **但它只估"阈值处的局部效应"**:RDD 得到的是**恰好卡在阈值那群人**的效应。奖学金对 60 分学生的效应,不代表它对 90 分学霸或 40 分学困生的效应(**外推性差**——这和 IV 的 LATE 同病相怜)。所以 RDD 的结论要说清楚"这是对阈值附近人群的效应"。
3. **命门是"无操纵"**:如果人们能**精确操纵**驱动变量卡到阈值(如老师给差一点的学生"送"到 60 分及格,或企业把利润做到刚好达标),阈值两侧就不再可比,RDD 崩溃。**必做 McCrary 密度检验**——看驱动变量的样本密度在阈值处有没有异常"堆积/跳变"(有跳变=有人在操纵)。本例密度连续(右图),假设成立。还要做**协变量连续性检验**(阈值处其他变量不应跳变)。**带宽选择**也很关键,要报告多个带宽下结果的稳健性。

**English**:
1. **RDD is the observational method closest to a randomized experiment**: the naive comparison is off at +30 (confounded by score's own effect), while RDD exactly captures the +8 jump at the cutoff. Its credibility rests on an irrefutable logic — **people scoring 59 vs 61 have virtually no systematic difference**, the only distinction being treatment. So **locally** at the cutoff, treatment is near-randomly assigned. This gives RDD very high internal validity, highly trusted in policy evaluation, education, and financial regulation.
2. **But it estimates only "the local effect at the cutoff"**: RDD gives the effect for **exactly those at the threshold**. The scholarship's effect on 60-point students doesn't represent its effect on 90-point top students or 40-point struggling ones (**poor external validity** — the same ailment as IV's LATE). So RDD conclusions must state "this is the effect for the cutoff population."
3. **The crux is "no manipulation"**: if people can **precisely manipulate** the running variable to land at the cutoff (a teacher "bumping" a near-fail to a passing 60, or a firm engineering profit to just meet a target), the two sides are no longer comparable and RDD collapses. **Always run the McCrary density test** — check for an abnormal "pile-up / jump" in the running variable's density at the cutoff (a jump = manipulation). Here density is continuous (right plot), so the assumption holds. Also run **covariate continuity tests** (other variables shouldn't jump at the cutoff). **Bandwidth choice** matters too — report robustness across several bandwidths.

> 💼 **实战视角 / Practical angle**
> **中文**:RDD 在**互联网/风控**里也很实用:①**规则触发的处理**——信用分≥某线自动提额、消费满额自动升级会员、用户评分≥阈值触发某策略, 都能用 RDD 估这些规则的真实效果;②广告/推荐里的排序阈值。落地要点:①**画断点图**(最直观的证据);②**局部线性/多项式回归 + 最优带宽**(如 IK/CCT 带宽);③**必做 McCrary 密度检验 + 协变量连续性检验**;④报告多带宽稳健性;⑤模糊断点用 IV 版本。工具:`rdrobust`(R/Python)。面试金句:*"RDD 利用阈值分配→阈值两侧 as-good-as-random, 效应=结果在阈值处的跳变(局部线性回归量落差); 假设是连续性/无操纵(McCrary 检验), 内部效度近 RCT 但只估阈值处的局部效应。"*
> **English**: RDD is also practical in **internet/risk**: ① **rule-triggered treatments** — credit score ≥ a line auto-raises limits, spend ≥ a threshold auto-upgrades membership, a user rating ≥ a cutoff triggers a policy — all estimable via RDD; ② ranking thresholds in ads/recommendation. Deployment keys: ① **draw the discontinuity plot** (the most intuitive evidence); ② **local linear/polynomial regression + optimal bandwidth** (e.g. IK/CCT); ③ **always run the McCrary density test + covariate continuity tests**; ④ report robustness across bandwidths; ⑤ use the IV version for fuzzy RDD. Tool: `rdrobust` (R/Python). Interview line: *"RDD exploits threshold assignment → both sides of the cutoff are as-good-as-random; the effect = the outcome's jump at the cutoff (local linear regression measures the gap); the assumption is continuity / no manipulation (McCrary test); near-RCT internal validity but only the local effect at the cutoff."*

---
### 小结 / Summary
- **中文**:RDD=按阈值分配处理→阈值两侧 as-good-as-random; 效应=结果在阈值处的跳变(局部线性回归)。
- **English**: RDD = threshold-assigned treatment → both sides as-good-as-random; effect = the outcome's jump at the cutoff (local linear regression).
- **中文**:Sharp(确定性)vs Fuzzy(改变处理概率→当IV); 带宽选择是偏差-方差权衡。
- **English**: Sharp (deterministic) vs Fuzzy (changes treatment probability → use as IV); bandwidth choice is a bias-variance trade-off.
- **中文**:核心假设=连续性/无操纵(McCrary 密度检验); 内部效度近 RCT 但只估阈值处局部效应。
- **English**: Core assumption = continuity / no manipulation (McCrary density test); near-RCT internal validity but only the local effect at the cutoff.
